# EDA 03 — Limites Administratives des Arrondissements
**Source** : Paris Open Data API Explore v2.1 | **Bronze** : `data/bronze/boundaries/`

In [ ]:

import os, glob, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
warnings.filterwarnings("ignore")

PALETTE = {
    "primary":   "#1565C0",
    "secondary": "#E53935",
    "accent":    "#2E7D32",
    "neutral":   "#546E7A",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
})

BRONZE_ROOT = os.path.abspath("../../data/bronze")
NB_DIR      = os.path.abspath(".")
FIGURES_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(name, source_prefix, tight=True):
    dest = os.path.join(FIGURES_DIR, source_prefix)
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, f"{name}.png")
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved -> figures/{source_prefix}/{name}.png")

print("Setup OK — figures ->", FIGURES_DIR)


In [ ]:

def draw_schema(title, columns, source_prefix, filename="schema"):
    n_rows = len(columns)
    fig_h  = max(3.0, 0.48 * n_rows + 1.6)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.99, ha="center")
    headers = ["Colonne", "Type", "Description"]
    col_x   = [0.0, 0.22, 0.37]
    col_w   = [0.22, 0.15, 0.63]
    row_h   = 1.0 / (n_rows + 1)
    for i, (hdr, x, w) in enumerate(zip(headers, col_x, col_w)):
        rect = mpatches.FancyBboxPatch(
            (x, 1 - row_h), w - 0.006, row_h * 0.88,
            boxstyle="round,pad=0.01", linewidth=0.5,
            edgecolor="#90A4AE", facecolor="#1565C0",
            transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)
        ax.text(x + w/2, 1 - row_h/2, hdr,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")
    type_colors = {
        "str":"#1565C0","int":"#2E7D32","float":"#6A1B9A",
        "datetime":"#E65100","bool":"#AD1457","date":"#00695C",
    }
    for ridx, (col_name, col_type, col_desc) in enumerate(columns):
        y_top = 1 - (ridx + 2) * row_h
        bg = "#F5F7FA" if ridx % 2 == 0 else "white"
        for x, w in zip(col_x, col_w):
            rect = mpatches.FancyBboxPatch(
                (x, y_top), w - 0.006, row_h * 0.88,
                boxstyle="round,pad=0.01", linewidth=0.5,
                edgecolor="#CFD8DC", facecolor=bg,
                transform=ax.transAxes, clip_on=False)
            ax.add_patch(rect)
        y_mid = y_top + row_h * 0.44
        tc = type_colors.get(col_type.split("[")[0].lower(), "#37474F")
        ax.text(col_x[0]+col_w[0]/2, y_mid, col_name,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, fontweight="bold", color="#1A237E")
        ax.text(col_x[1]+col_w[1]/2, y_mid, col_type,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9, fontfamily="monospace", color=tc)
        ax.text(col_x[2]+0.01, y_mid, col_desc,
                transform=ax.transAxes, ha="left", va="center",
                fontsize=9.5, color="#37474F")
    plt.subplots_adjust(top=0.92, bottom=0)
    save_fig(filename, source_prefix)
    plt.show()


In [ ]:
PREFIX = "03_boundaries"
draw_schema(
    "Bronze Schema — Limites Administratives (Arrondissements Parisiens)",
    [
        ("arrondissement","int",      "Numero d'arrondissement (1-20)"),
        ("c_ar",          "str",      "Code court arrondissement"),
        ("c_arinsee",     "str",      "Code INSEE commune (751xx)"),
        ("l_ar",          "str",      "Libelle officiel (ex: 12eme Arrondissement)"),
        ("surface_ha",    "float",    "Superficie en m2 (unite source)"),
        ("centroid_lat",  "float",    "Latitude du centroide"),
        ("centroid_lon",  "float",    "Longitude du centroide"),
        ("geometry_wkt",  "str",      "Polygone WKT en EPSG:4326 (utilise pour les jointures spatiales Silver)"),
        ("ingested_at",   "datetime", "Horodatage UTC d'ingestion"),
    ], PREFIX
)

In [ ]:
path = f"{BRONZE_ROOT}/boundaries/part-0.parquet"
df = pd.read_parquet(path) if os.path.exists(path) else pd.DataFrame()
if not df.empty:
    df["surface_km2"] = df["surface_ha"] / 1e6
    print(f"Shape : {df.shape}")
    display(df[["arrondissement","l_ar","surface_km2","centroid_lat","centroid_lon"]].sort_values("arrondissement"))
else:
    print("Fichier absent")

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1,2,figsize=(16,6))
    df_s = df.sort_values("arrondissement")
    norm = df_s["surface_km2"] / df_s["surface_km2"].max()
    axes[0].bar(df_s["arrondissement"].astype(str), df_s["surface_km2"], color=plt.cm.Blues(0.3+norm*0.6), edgecolor="white")
    axes[0].set_xlabel("Arrondissement"); axes[0].set_ylabel("Superficie (km2)")
    axes[0].set_title("Superficie des 20 arrondissements parisiens")
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)
    sm = plt.cm.ScalarMappable(cmap="Blues",norm=plt.Normalize(df_s["surface_km2"].min(),df_s["surface_km2"].max()))
    plt.colorbar(sm,ax=axes[0],label="km2")
    sc = axes[1].scatter(df_s["centroid_lon"],df_s["centroid_lat"],c=df_s["surface_km2"],cmap="Blues",s=400,
                          zorder=5,edgecolors="#1565C0",linewidths=1.5)
    for _, row in df_s.iterrows():
        axes[1].annotate(str(row["arrondissement"]),(row["centroid_lon"],row["centroid_lat"]),
                         ha="center",va="center",fontsize=8,fontweight="bold",color="white")
    plt.colorbar(sc,ax=axes[1],label="km2")
    axes[1].set_title("Centroides (taille & couleur = superficie)")
    axes[1].set_xlabel("Longitude"); axes[1].set_ylabel("Latitude"); axes[1].set_aspect("equal")
    save_fig("superficie_et_centroides", PREFIX)
    plt.show()
    print(f"Plus grand  : {df_s.nlargest(1,'surface_km2').iloc[0]['l_ar']} — {df_s['surface_km2'].max():.2f} km2")
    print(f"Plus petit  : {df_s.nsmallest(1,'surface_km2').iloc[0]['l_ar']} — {df_s['surface_km2'].min():.2f} km2")
    print(f"Total Paris : {df_s['surface_km2'].sum():.2f} km2")

In [ ]:
if not df.empty:
    try:
        from shapely import wkt
        import matplotlib.cm as cm
        fig, ax = plt.subplots(figsize=(10,12))
        cmap = cm.get_cmap("tab20")
        for _, row in df.iterrows():
            poly = wkt.loads(row["geometry_wkt"])
            x, y = poly.exterior.xy
            color = cmap(row["arrondissement"]/21)
            ax.fill(x,y,alpha=0.72,color=color); ax.plot(x,y,color="white",linewidth=1.2)
            cx,cy = poly.centroid.x, poly.centroid.y
            ax.text(cx,cy,str(row["arrondissement"]),ha="center",va="center",fontsize=9,fontweight="bold",
                    color="white",bbox=dict(boxstyle="round,pad=0.15",facecolor="#1565C0",alpha=0.75,edgecolor="none"))
        ax.set_title("Carte des 20 arrondissements parisiens",fontsize=14)
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.set_aspect("equal")
        save_fig("carte_arrondissements", PREFIX)
        plt.show()
    except ImportError:
        print("pip install shapely  pour la carte polygonale")